# IterCOMP - Tái hiện trên Colab / Kaggle T4

Tái hiện **IterCOMP** ([ACL 2026](https://aclanthology.org/2026.acl-long.1559/))
trên 4 bộ dữ liệu + mở rộng cho tiếng Việt.
Bài báo gốc không công bố mã nguồn lẫn prompt - prompt do nhóm thiết kế.

### Chạy

1. Bật GPU T4 - Colab: `Runtime` → `Change runtime type`. Kaggle: `Settings` → `Accelerator`.
2. Chọn cỡ mẫu ở ô §3 (biến `SCALE`) - xem bảng dưới.
3. `Run all`.

Notebook tự tải mã nguồn và dữ liệu, không cần upload gì.

### Cỡ mẫu - sửa `SCALE` ở ô §3

| `SCALE` | Dữ liệu | Thời gian | Dùng khi |
|---|---|---|---|
| `'smoke'` | n=50 | ~0,3h | Kiểm tra chạy được. Số **không kết luận được** (dưới ngưỡng 18,0 F1) |
| `'paper'` | VimQA full · MuSiQue 500+1500 · HotpotQA/2Wiki 1500 | ~16h | Mặc định - **đúng quy mô sinh số trong báo cáo** |
| `'max'` | thêm MuSiQue full | ~18h | |

Nếu ô §1 báo `Ghim KHÔNG ăn`: `Restart session` rồi `Run all` lại.
Phiên đứt giữa chừng không mất kết quả - có checkpoint sau mỗi câu.


In [ ]:
import torch
p = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None

if p is None:
    # Không có GPU vẫn chạy được các ô nhẹ: khám phá dữ liệu (2b) và quét k
    # (3c) chỉ cần bge-m3, vốn chạy CPU được. Bảng chính và ablation thì bắt
    # buộc GPU vì phải nạp mô hình đọc 7B.
    GPU = False
    SUGGEST = None
    print('KHÔNG có GPU - chỉ chạy được ô 2b (khám phá) và 3c (quét k).')
    print('Bảng chính và ablation cần GPU: Runtime > Change runtime type > T4.')
else:
    GPU = True
    maj, mnr = torch.cuda.get_device_capability(0)
    gb = p.total_memory / 1e9
    print(f'{p.name} | {gb:.1f} GB | sm_{maj}{mnr}')
    assert maj >= 7, f'GPU sm_{maj}{mnr} không chạy được PyTorch - chọn T4'

    # Chọn cỡ mô hình theo VRAM THẬT, không giả định là T4 15 GB.
    # Mô hình 7B ở 4-bit chiếm ~5,5 GB, cộng KV-cache cần ~6,5 GB trống.
    if gb >= 12:
        SUGGEST = 'Qwen/Qwen2.5-7B-Instruct'
        print(f'{gb:.0f} GB đủ cho 7B ở 4-bit')
    else:
        SUGGEST = 'Qwen/Qwen2.5-1.5B-Instruct'
        print(f'! Chỉ {gb:.0f} GB: KHÔNG đủ cho 7B. Ô cấu hình sẽ dùng 1.5B.')

## 1. Mã nguồn và thư viện

In [ ]:
# ══ GHIM transformers==4.45.2 ══
# Colab/Kaggle nay có thể ship transformers 5.0.0, mà `llmlingua 0.2.2` viết cho
# 4.x: 5.0 đòi `past_key_values` là `Cache` object, llmlingua lại lặp
# `for k, v in past_key_values`. Hai yêu cầu loại trừ nhau - sáu phiên GPU đã
# mất vì cố monkey-patch điều bất khả. Ghim là bản vá thật, và 4.45.2 là đúng
# phiên bản mọi kết quả trong báo cáo được sinh ra.
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. '
    f'Runtime > Restart session rồi chạy lại từ đầu.')


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean
print('cwd:', os.getcwd(), '| src/scripts/:', os.path.isdir('src/scripts'))
print(f'đủ {len(NEED)} module')

## 2. Dữ liệu

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
from itercomp import load_dataset
for ds in ('hotpotqa', '2wiki', 'musique', 'vimqa'):
    r = load_dataset(ds, 1)[0]
    n = len(r.get('supporting_facts', {}).get('title', []))
    print(f"{ds:9s} {len(r['context']['title']):2d} đoạn | supporting_facts={n}")

## 2b. Khám phá và tiền xử lý dữ liệu

In [ ]:
# ══ 2b. KHÁM PHÁ VÀ TIỀN XỬ LÝ DỮ LIỆU ══
import statistics, collections
from itercomp import load_dataset
from itercomp.core import decompose
from itercomp.metrics import tokenize

rows = load_dataset('vimqa', 200)

# ── thống kê mô tả ─────────────────────────────────────────────────
n_para = [len(r['context']['title']) for r in rows]
n_tok  = [len(tokenize('\n'.join(' '.join(s) for s in r['context']['sentences'])))
          for r in rows]
n_gold = [len(r.get('supporting_facts', {}).get('title', [])) for r in rows]

print(f'VimQA - {len(rows)} câu hỏi')
print(f'  đoạn/câu     : trung vị {statistics.median(n_para):.0f}'
      f'  (min {min(n_para)}, max {max(n_para)})')
print(f'  token/ngữ cảnh: trung vị {statistics.median(n_tok):.0f}'
      f'  (min {min(n_tok)}, max {max(n_tok)})')
print(f'  câu bằng chứng vàng/câu hỏi: trung vị {statistics.median(n_gold):.0f}')

# ── phân bố loại câu hỏi: đúng/sai chiếm bao nhiêu? ────────────────
BOOL = {'đúng', 'không', 'sai', 'có'}
kind = collections.Counter('đúng/sai' if str(r['answer']).strip().lower() in BOOL
                           else 'trích xuất' for r in rows)
print('\nloại đáp án:')
for k, v in kind.most_common():
    print(f'  {k:12s} {v:3d}  {v/len(rows):5.1%}')
print('  -> nhóm đúng/sai lớn nên chuẩn hoá đáp án ảnh hưởng đáng kể')

# ── tiền xử lý: phân rã tài liệu thành segment bằng chứng ──────────
r = rows[0]
segs = decompose(r['context'])
print(f'\nTIỀN XỬ LÝ - phân rã câu hỏi đầu tiên:')
print(f'  {len(r["context"]["title"])} đoạn  ->  {len(segs)} segment câu')
for s in segs[:3]:
    print(f'    [{s.doc_title[:22]:24s}] {str(s)[:58]}…')

# ── kiểm tra tokenizer giữ dấu tiếng Việt ──────────────────────────
mau = 'Album được phát hành vào ngày 29 tháng 3 năm 2019'
print(f'\nTOKENIZER (bắt buộc dùng re.UNICODE cho tiếng Việt):')
print(f'  vào  -> {tokenize("vào")}')
print(f'  má/mà/mã giữ nguyên dấu: {tokenize("má mà mã")}')
assert tokenize('vào') == ['vào'], 'tokenizer làm hỏng dấu tiếng Việt'
print('  dấu thanh được giữ nguyên')

## 3. Cấu hình, rồi kiểm tra vòng lặp trước khi chạy dài

In [ ]:
import subprocess, time, os, sys, glob, json

# ══ MÔ HÌNH ĐỌC ══
# Bài báo dùng LLaMA-3-8B. Qwen2.5-7B mở hoàn toàn và cùng bậc năng lực.
READER = SUGGEST or 'Qwen/Qwen2.5-7B-Instruct'  # None khi chạy CPU
LOAD_4BIT = True     # bắt buộc với 7-8B trên T4 15GB
TAG = 'hf'           # backend suy luận

# ══ QUY MÔ ══
# Chọn một bậc. Chi phí suy ra từ s/câu ĐÃ ĐO trên T4 (Qwen2.5-7B 4-bit),
# tổng 4 phương pháp/câu: VimQA 9,6 · MuSiQue 15 · HotpotQA 6,5 · 2Wiki 6,2.
#
#   bậc      VimQA      MuSiQue      HotpotQA/2Wiki   tổng   ngưỡng phát hiện
#   'smoke'  n=50       n=50         -                0,3h   18,0 F1 (không kết luận)
#   'paper'  FULL 1003  500 + 1500   1500 + 1500      ~16h   4,3 F1 (VimQA full) ← mặc định
#   'max'    FULL 1003  FULL 2417    -                ~18h   4,3 F1
#
# Vì sao 'paper' mặc định: ĐÚNG quy mô sinh số báo cáo — VimQA full (n=1003),
# MuSiQue n=500 (bảng chính) kèm n=1500 (khẳng định §5), HotpotQA/2Wiki n=1500
# (bảng 4 dataset). Ở n=50 ngưỡng phát hiện 18,0 F1 nên số 'smoke' không kết
# luận được; VimQA full hạ ngưỡng xuống 4,3 F1.
#
# Vì sao KHÔNG full cả 4 bộ ở 'paper': HotpotQA (7405) + 2Wiki (12576) full
# ~36h. n=1500 đã đủ cho mọi khẳng định (thứ tự phương pháp, CI cắt-0).
SCALE = 'paper'

SCALES = {
    'smoke': {'vimqa': 50,   'musique': 50},
    # 'paper' = đúng quy mô báo cáo: MuSiQue cả 500 (bảng chính) lẫn 1500 (§5),
    # HotpotQA/2Wiki ở 1500 (bảng 4 dataset).
    'paper': {'vimqa': None, 'musique': 500, 'hotpotqa': 1500, '2wiki': 1500},
    'max':   {'vimqa': None, 'musique': None},
}
assert SCALE in SCALES, f'SCALE phải thuộc {list(SCALES)}'
PLAN = SCALES[SCALE]          # None = chạy TOÀN BỘ tập dev

# MuSiQue cần THÊM n=1500 cho khẳng định §5 (bảng chính vẫn dùng n=500). Đây
# là lần chạy ĐẮT NHẤT (~6h); muốn nhanh, đặt RUN_MUSIQUE_1500 = False thì
# notebook vẫn tái hiện đủ bảng chính, chỉ thiếu CI cắt-0 của §5.
RUN_MUSIQUE_1500 = True
EXTRA_RUNS = ([('musique', 1500)]
              if RUN_MUSIQUE_1500 and SCALE in ('paper', 'max') else [])

# Hậu tố tên file lấy từ MÔ HÌNH THẬT, không cứng '7b'. Nếu GPU không đủ và ô
# kiểm tra lùi xuống 1.5B thì kết quả phải mang tên 1.5b - nếu không, số 1.5B
# nằm trong file tên _7b và sẽ bị chép vào báo cáo như số 7B.
MTAG = READER.split('/')[-1].replace('Qwen2.5-', '').replace('-Instruct', '').lower()

def nlabel(n):
    """Nhãn cỡ mẫu dùng trong tên file. 'full' rõ nghĩa hơn con số."""
    return 'full' if n is None else str(n)

def outfile(ds, n, prefix=''):
    return f'results/{prefix}{ds}_{nlabel(n)}_{MTAG}.json'

# Muốn dùng đúng LLaMA-3-8B: xin quyền tại
# https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct rồi bỏ chú thích
# from huggingface_hub import login
# login()
# READER = 'meta-llama/Meta-Llama-3-8B-Instruct'

os.makedirs('results', exist_ok=True)

# Kết quả cũ đo bằng '--itercomp-llm mock', trong đó vòng lặp KHÔNG hề chạy
# (mock luôn báo "đủ bằng chứng" ngay vòng 1). Chúng không so được với bài báo.
for f in glob.glob('results/*_7b.json'):
    old = f.replace('_7b.json', '_7b_mock.json')
    if not os.path.exists(old):
        os.rename(f, old)
        print('đổi tên (kết quả mock cũ):', os.path.basename(f), '->', os.path.basename(old))

# Ước thời gian để biết trước có vừa phiên Colab hay không.
SEC_PER_Q = {'vimqa': 9.6, 'musique': 15.0, 'hotpotqa': 6.5, '2wiki': 6.2}  # đo thật T4
FULL_N = {'vimqa': 1003, 'musique': 2417, 'hotpotqa': 7405, '2wiki': 12576}
tot = 0.0
print(f'reader = {READER} | 4-bit = {LOAD_4BIT} | SCALE = {SCALE!r}\n')
print(f"{'dataset':10s} {'n':>7s} {'ước tính':>10s}")
for ds, n in list(PLAN.items()) + EXTRA_RUNS:
    eff = FULL_N[ds] if n is None else n
    h = eff * SEC_PER_Q[ds] / 3600
    tot += h
    print(f'{ds:10s} {nlabel(n):>7s} {h:9.1f}h')
print(f"{'TỔNG':10s} {'':>7s} {tot:9.1f}h"
      + ('  vượt một phiên Colab 12h' if tot > 12 else ''))


In [ ]:
# ══ BẰNG CHỨNG: vòng lặp có thật sự chạy hay không ══
# Chạy ô này TRƯỚC bảng chính. Nó chỉ mất vài phút và cho biết ngay các bước
# suy luận có hoạt động không - khỏi chờ hết phiên mới phát hiện sai. Bảng này
# vào Phụ lục báo cáo: nó cho thấy 'mock' làm IterCOMP suy thoái thành "lọc
# một lần" vì luôn trả ANSWERABLE_YES ở vòng 1.
#
# VRAM: mỗi tag chạy trong TIẾN TRÌNH RIÊNG. Giữ scorer (bge-m3, ~2,3 GB) và
# reader 7B 4-bit (~5,5 GB + KV-cache) trong cùng một tiến trình vừa khít Colab
# T4 (15,0 GB) nhưng KHÔNG vừa Kaggle T4 (14,56 GB). Và gọi gc + empty_cache
# KHÔNG đủ: buffer 4-bit của bitsandbytes chỉ thực sự trả về cho driver khi
# tiến trình thoát. Nên tách tiến trình thay vì cố dọn trong tiến trình.
import json, subprocess, sys, textwrap

EVID = textwrap.dedent("""
    import json, sys
    sys.path.insert(0, 'src')
    from itercomp import load_dataset, itercomp, make_llm, make_scorer
    from itercomp.metrics import tokenize

    tag, reader, four = sys.argv[1], sys.argv[2], sys.argv[3] == '1'
    sc = make_scorer('dual')
    llm = make_llm('mock') if tag == 'mock' else make_llm(
        'hf', model=reader, load_4bit=four)

    it = nz = 0.0
    why = {}
    n = 0
    for r in load_dataset('vimqa', 10):
        res = itercomp(llm, r['question'], r['context'],
                       max_iter=5, scorer=sc, percentile=90)
        full = '\\n'.join(f'{t}: {" ".join(s)}'
                          for t, s in zip(r['context']['title'],
                                          r['context']['sentences']))
        it += res.iterations
        nz += len(tokenize(res.to_prompt())) / max(len(tokenize(full)), 1)
        why[res.stopped_because] = why.get(res.stopped_because, 0) + 1
        n += 1
    print('RESULT ' + json.dumps([tag, it / n, nz / n * 100, why]))
""")

rows_ev = []
for tag in ['mock', 'hf']:
    print(f'--- {tag} ---', flush=True)
    p = subprocess.run([sys.executable, '-c', EVID, tag, READER,
                        '1' if LOAD_4BIT else '0'],
                       capture_output=True, text=True)
    line = next((l for l in p.stdout.splitlines()
                 if l.startswith('RESULT ')), None)
    if line is None:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
        raise SystemExit(f'tag {tag} không trả kết quả')
    rows_ev.append(json.loads(line[len('RESULT '):]))
    print(f'  vòng={rows_ev[-1][1]:.2f}  tỉ lệ={rows_ev[-1][2]:.1f}%', flush=True)

import os
os.makedirs('results', exist_ok=True)
json.dump([{'tag': t, 'iters': i, 'ratio': z, 'stop_reason': w}
           for t, i, z, w in rows_ev],
          open('results/loop_evidence.json', 'w'), ensure_ascii=False, indent=1)

print()
print(f"{'backend':10s}{'vòng TB':>9s}{'tỉ lệ nén':>11s}  lý do dừng")
for t, i, z, w in rows_ev:
    print(f'{t:10s}{i:9.2f}{z:10.1f}%  {w}')

assert rows_ev[1][1] > rows_ev[0][1], \
    'hf không lặp nhiều vòng hơn mock -> backend hf chưa hoạt động'
print('\nOK: backend hf lặp nhiều vòng hơn mock.')

# Không cần dọn VRAM: cả hai tag đã chạy trong tiến trình con và thoát rồi.
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM trống: {free/1e9:.1f} / {total/1e9:.1f} GB - chạy tiếp bảng chính.')


## 3b. Bảng chính - số để so với bài báo

## 3c. Quét $k$ để khớp tỉ lệ nén của bài báo

In [ ]:
# ══ QUÉT k ĐỂ KHỚP TỈ LỆ NÉN CỦA BÀI BÁO ══
# Bài báo báo tỉ lệ nén 0.14 trên MuSiQue với k=90; ta được 0.32 ở cùng k.
# Chênh này là khác biệt HÀNH VI, không phải nhiễu - tăng cỡ mẫu không xoá được.
# Tỉ lệ nén chỉ phụ thuộc scorer + ngưỡng, KHÔNG phụ thuộc mô hình đọc, nên ô
# này CHẠY ĐƯỢC KHÔNG CẦN GPU - bge-m3 chấm điểm trên CPU vẫn được, chỉ chậm
# hơn. Hữu ích khi đã hết quota GPU.
import sys; sys.path.insert(0, 'src')
from itercomp import load_dataset, itercomp, make_scorer, make_llm
from itercomp.metrics import tokenize

N_SWEEP = 50
sc, llm = make_scorer('dual'), make_llm('mock')

for ds, target in (('musique', 0.14), ('vimqa', None)):
    rows = load_dataset(ds, N_SWEEP)
    print(f'\n{ds}  (n={len(rows)})' + (f'  - mục tiêu {target:.2f}' if target else ''))
    print(f"  {'k':>5}{'tỉ lệ nén':>12}{'lệch':>9}")
    best = None
    for k in (88, 90, 92, 94, 95, 96, 97):
        tot = 0.0
        for r in rows:
            res = itercomp(llm, r['question'], r['context'],
                           max_iter=5, scorer=sc, percentile=k)
            full = '\n'.join(f'{t}: {" ".join(s)}' for t, s
                             in zip(r['context']['title'], r['context']['sentences']))
            tot += len(tokenize(res.to_prompt())) / max(len(tokenize(full)), 1)
        ratio = tot / len(rows)
        mark = ''
        if target:
            d = abs(ratio - target)
            if best is None or d < best[1]:
                best, mark = (k, d), ''
            print(f'  {k:>5}{ratio*100:11.1f}%{(ratio-target)*100:+8.1f}pp')
        else:
            print(f'  {k:>5}{ratio*100:11.1f}%{"":>9}')
    if target:
        print(f'  -> k={best[0]} bám sát {target:.2f} nhất')

print('\nDùng k tìm được để chạy lại bảng chính; khi đó tỉ lệ nén so được với')
print('bài báo, và chênh F1 còn lại quy về mô hình đọc và prompt.')

In [ ]:
# ══ BẢNG CHÍNH ══
# Cỡ mẫu và danh sách bộ dữ liệu lấy từ PLAN (ô cấu hình), không cứng ở đây.
#
# Bỏ qua bộ đã có file kết quả; trong mỗi bộ, run_eval.py còn ghi checkpoint
# sau MỖI câu, nên tắt máy giữa chừng chạy lại vẫn tiếp từ chỗ dở. Với bậc
# 'paper' (VimQA full) tính năng này là bắt buộc chứ không phải tiện lợi.

def run_stream(cmd, logfile):
    """Chạy và in output NGAY, đồng thời lưu log.

    Không dùng capture_output=True: 7B ở 4-bit chạy hàng giờ, giữ hết output
    tới lúc xong nghĩa là ngồi nhìn màn hình trống, không biết treo hay đang
    chạy - và Colab hay tự ngắt khi cell im lặng quá lâu. Ghi log ra file để
    nếu bị ngắt giữa dataset vẫn còn vết.
    """
    tail = []
    with open(logfile, 'w') as f:
        pr = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in pr.stdout:
            print(line, end='', flush=True)
            f.write(line)
            tail.append(line)
            if len(tail) > 400:
                del tail[0]
        pr.wait()
    return pr.returncode, ''.join(tail)

# VimQA trước: đó là phần đóng góp tiếng Việt và là bộ rẻ nhất để chạy full,
# nên nếu phiên bị ngắt thì phần quan trọng nhất đã xong.
ORDER = ['vimqa', 'musique', 'hotpotqa', '2wiki']
# (dataset, n): các bộ trong PLAN + lần chạy MuSiQue n=1500 phụ (§5).
RUNS = [(d, PLAN[d]) for d in ORDER if d in PLAN] + EXTRA_RUNS
for ds, n in RUNS:
    out = outfile(ds, n)
    if os.path.exists(out):
        print(f'[bỏ qua] {ds} đã có kết quả ({out})')
        continue
    print('\n' + '=' * 70)
    print(f'### {ds}  n={nlabel(n)}   bắt đầu {time.strftime("%H:%M")}')
    print('=' * 70, flush=True)
    t0 = time.time()
    rc, _ = run_stream(
        [sys.executable, '-u', 'src/scripts/run_eval.py', '--dataset', ds,
         # bỏ --limit hoàn toàn = chạy TOÀN BỘ tập (run_eval mặc định là full)
         *(['--limit', str(n)] if n is not None else []),
         '--reader', 'hf', '--reader-model', READER,
         *(['--load-4bit'] if LOAD_4BIT else []),
         '--methods', 'raw,oracle,llmlingua2,itercomp',
         # Các bước suy luận (phân rã / kiểm tra đủ bằng chứng / sinh câu hỏi
         # tiếp) phải dùng LLM thật như bài báo, không phải mock.
         '--itercomp-llm', TAG, '--scorer', 'dual', '--out', out],
        f'results/log_{ds}_{nlabel(n)}_{MTAG}.txt')
    if rc:
        print(f'!!! {ds} LỖI (mã {rc}) - xem log')
    elif not os.path.exists(out):
        # Thoát mã 0 mà không sinh file nghĩa là có gì đó sai ngầm.
        print(f'!!! {ds} chạy xong nhưng KHÔNG có {out}')
    print(f'--> {(time.time() - t0) / 60:.1f} phút', flush=True)

print('\nXONG bảng chính.')


## 4. Ablation - 26 cấu hình, có EM/F1

In [ ]:
# Chạy SAU khi đã có bảng chính (cần run_stream định nghĩa ở ô bảng chính).
#
# Ablation quét 26 cấu hình nên tốn gấp nhiều lần bảng chính. Giữ n=200 ngay cả
# ở bậc 'paper': ablation dùng để so các cấu hình VỚI NHAU trên cùng cỡ mẫu,
# và ngưỡng phát hiện ở n=200 đủ để thấy trục nào quan trọng. Chạy
# full 1.003 câu × 26 cấu hình sẽ mất ~70h mà không đổi kết luận nào.
ABL_N = 200 if SCALE != 'smoke' else 50

for ds in ['vimqa']:
    out = f'results/ablation_{ds}_{ABL_N}_{MTAG}_{TAG}.json'
    if os.path.exists(out):
        print(f'[bỏ qua] {ds} đã có'); continue
    print('\n' + '='*70 + f'\n### Ablation - {ds}  n={ABL_N}  (26 cấu hình)\n' + '='*70, flush=True)
    t0 = time.time()
    rc, _ = run_stream([sys.executable,'-u','src/scripts/run_ablation.py','--dataset',ds,
        '--limit',str(ABL_N),'--llm',TAG,'--reader','hf',
        '--reader-model',READER, *(['--load-4bit'] if LOAD_4BIT else []),
        '--out',out], f'results/log_ablation_{ds}_{ABL_N}_{MTAG}.txt')
    if rc:
        print(f'!!! ablation {ds} LỖI (mã {rc})')
    elif not os.path.exists(out):
        print(f'!!! ablation {ds} xong nhưng KHÔNG có {out}')
    print(f'--> {(time.time()-t0)/60:.1f} phút', flush=True)


## 7. Phân tích lỗi

Phân loại lỗi của IterCOMP trên VimQA từ `per_row` đã chạy ở ô Bảng chính (không suy luận lại). Tái hiện §Error Analysis của báo cáo: bốn nhóm theo F1, lỗi định dạng yes/no, và mẫu câu so-sánh tuổi.

In [ ]:
# ══ 7. PHÂN TÍCH LỖI ══
# Đọc lại per_row VimQA (ô Bảng chính đã ghi) - không suy luận lại.
# Dùng F1* = f1_norm (đã chuẩn hoá boolean), đúng metric của báo cáo.
import glob, json

cand = [f for f in sorted(glob.glob(f'results/vimqa_*_{MTAG}.json'))
        if 'ablation' not in f and json.load(open(f)).get('per_row')]
assert cand, 'Chưa có kết quả VimQA - chạy ô "Bảng chính" trước.'
rows = json.load(open(cand[-1]))['per_row']
n = len(rows)
print(f'Phân tích {n} câu VimQA  ({cand[-1]})\n')

def m(r): return r['methods']['itercomp']            # F1* = f1_norm
b = {'Đúng hoàn toàn (F1*=1)': 0, 'Đúng một phần (0<F1*<1)': 0,
     'Sai hoàn toàn (F1*=0)': 0}
fmt = 0                                               # lỗi bề mặt yes/no
for r in rows:
    fn, fr = m(r)['f1_norm'], m(r)['f1']
    if fn == 1:   b['Đúng hoàn toàn (F1*=1)'] += 1
    elif fn > 0:  b['Đúng một phần (0<F1*<1)'] += 1
    else:         b['Sai hoàn toàn (F1*=0)'] += 1
    if fr == 0 and fn == 1: fmt += 1                  # thô sai, chuẩn hoá cứu

print('Phân bố lỗi IterCOMP (theo F1*):')
for k, v in b.items():
    print(f'  {k:<26} {v:>4}  ({100*v/n:4.0f}%)')
print(f'\nLỗi định dạng yes/no được boolean-norm cứu (F1 thô=0 → F1*=1): '
      f'{fmt} ({100*fmt/n:.0f}%) - chính là đóng góp của chuẩn hoá boolean.')

# Mẫu so-sánh tuổi: số học chứ không phải truy hồi
age = [r for r in rows if any(p in r['question'].lower()
       for p in ('tuổi hơn', 'già hơn', 'trẻ hơn', 'lớn tuổi', 'nhỏ tuổi'))]
if age:
    z  = sum(1 for r in age if m(r)['f1_norm'] == 0)
    rest = [r for r in rows if r not in age]
    zr = sum(1 for r in rest if m(r)['f1_norm'] == 0)
    print(f'\nMẫu so-sánh tuổi: {len(age)}/{n} câu ({100*len(age)/n:.1f}%); '
          f'{100*z/len(age):.1f}% bị F1*=0 so với {100*zr/len(rest):.1f}% ở phần còn '
          f'lại - câu hỏi số học, không phải truy hồi.')


## 8. Demo trên dữ liệu tiếng Việt mới

In [ ]:
# ══ 8. DEMO TRÊN DỮ LIỆU TIẾNG VIỆT MỚI ══
# Câu hỏi và ngữ cảnh TỰ VIẾT, không có trong VimQA - kiểm tra hệ thống
# hoạt động trên dữ liệu tiếng Việt chưa từng thấy.
from itercomp import itercomp, make_scorer, make_llm
from itercomp.metrics import f1_score, normalize_boolean

ngu_canh = {
    'title': ['Hồ Chí Minh', 'Hồ Chí Minh', 'Hà Nội', 'Đà Nẵng', 'Huế'],
    'sentences': [
        ['Thành phố Hồ Chí Minh là thành phố lớn nhất Việt Nam về dân số.'],
        ['Thành phố này trước năm 1976 có tên là Sài Gòn.'],
        ['Hà Nội là thủ đô của Việt Nam, nằm ở miền Bắc.'],
        ['Đà Nẵng là thành phố trực thuộc trung ương ở miền Trung.'],
        ['Huế từng là kinh đô của triều Nguyễn.'],
    ],
}
cau_hoi = 'Thành phố lớn nhất Việt Nam về dân số trước năm 1976 tên là gì?'
dap_an  = 'Sài Gòn'

print('CÂU HỎI :', cau_hoi)
print('ĐÁP ÁN  :', dap_an)
print(f'NGỮ CẢNH: {sum(len(s) for s in ngu_canh["sentences"])} câu\n')

sc  = make_scorer('dual')
llm = make_llm('hf', model=READER, load_4bit=LOAD_4BIT)
res = itercomp(llm, cau_hoi, ngu_canh, max_iter=5, scorer=sc, percentile=90)

print(f'--- IterCOMP: {res.iterations} vòng, dừng vì "{res.stopped_because}" ---')
for s in res.selected:
    print(f'  giữ: {str(s)[:70]}')

from itercomp import make_reader
rd = make_reader('hf', model=READER, load_4bit=LOAD_4BIT)
full = '\n'.join(f'{t}: {" ".join(s)}'
                 for t, s in zip(ngu_canh['title'], ngu_canh['sentences']))
for ten, ctx in (('raw (toàn bộ)', full), ('itercomp (đã nén)', res.to_prompt())):
    pred = rd(ctx, cau_hoi)
    f1 = f1_score(normalize_boolean(pred, dap_an), dap_an)
    print(f'\n{ten:20s} -> {pred[:50]!r}  F1={f1:.2f}')

print('\nĐây là câu multi-hop: phải nối "thành phố lớn nhất" với "tên trước 1976".')

## 10. EXP-1 - trần của việc dừng đúng lúc (CỬA CHẶN)

Chạy ô này **trước** mọi thí nghiệm nghiên cứu khác. Nó trả lời: nếu bộ dừng
hoàn hảo thì được thêm bao nhiêu điểm F1?

Nếu trần dưới 3 điểm thì cải thiện bộ dừng không đáng làm, và cả hướng nghiên
cứu dừng ở đây - đỡ hàng trăm giờ GPU. Chỉ tốn ~30 phút để biết.

In [ ]:
assert GPU, 'Ô này cần GPU'
import sys
# EXP-1 giữ n=200: nó so 4 CHẾ ĐỘ DỪNG trên cùng cỡ mẫu, nên chỉ cần đủ để
# thấy chênh lệch giữa các chế độ, không cần độ chính xác tuyệt đối.
EXP1_N = 200 if SCALE != 'smoke' else 50
for ds in ('vimqa', 'musique'):
    out = f'results/exp1_oracle_{ds}_{EXP1_N}_{MTAG}.json'
    if os.path.exists(out):
        print(f'[bỏ qua] EXP-1 {ds} đã có'); continue
    print('=' * 62)
    print(f'### EXP-1 - {ds}  n={EXP1_N}')
    print('=' * 62, flush=True)
    rc, _ = run_stream(
        [sys.executable, '-u', 'src/scripts/exp1_oracle_stopping.py',
         '--dataset', ds, '--limit', str(EXP1_N),
         '--reader', 'hf', '--reader-model', READER,
         *(['--load-4bit'] if LOAD_4BIT else []),
         '--scorer', 'dual', '--percentile', '90',
         '--out', out],
        f'results/log_exp1_{ds}_{EXP1_N}_{MTAG}.txt')
    if rc:
        print(f'!!! {ds} LỖI (mã {rc})')


## 11. EXP-2 - tiếng Việt KHÔNG DẤU

Lấp một limitation báo cáo tự nêu: chưa từng đánh giá tiếng Việt không dấu, dù
đó là dạng phổ biến trên web.

Ba chế độ dữ liệu, mỗi chế độ chạy thêm một nhánh gold-context để **tách nguyên
nhân**: phần gold mất là lỗi của mô hình đọc, phần IterCOMP mất THÊM so với gold
mới là lỗi của bộ nén. Đo F1 cuối một mình sẽ trộn hai nguyên nhân này.

In [ ]:
assert GPU, 'Ô này cần GPU'
# n=200 đủ để thấy hiệu ứng gỡ dấu (nếu có) vượt ngưỡng phát hiện; chạy full
# 1.003 × 3 chế độ × 2 nhánh sẽ mất ~16h mà câu hỏi ở đây là CÓ/KHÔNG.
EXP2_N = 200 if SCALE != 'smoke' else 50
out = f'results/exp2_undiacritised_{EXP2_N}_{MTAG}.json'
if os.path.exists(out):
    print(f'[bỏ qua] đã có {out}')
else:
    print('=' * 62)
    print(f'### EXP-2 - tiếng Việt không dấu  n={EXP2_N}')
    print('=' * 62, flush=True)
    rc, _ = run_stream(
        [sys.executable, '-u', 'src/scripts/exp2_undiacritised.py',
         '--limit', str(EXP2_N),
         '--reader', 'hf', '--reader-model', READER,
         *(['--load-4bit'] if LOAD_4BIT else []),
         '--scorer', 'dual', '--percentile', '90', '--out', out],
        f'results/log_exp2_{EXP2_N}_{MTAG}.txt')
    if rc:
        print(f'!!! EXP-2 LỖI (mã {rc})')


## 12. Quét $\lambda$ và $k$ trên VimQA

Báo cáo nêu một *kết quả âm*: $\lambda$ tốt nhất là 0,4 so với 0,6 của bài báo,
nhưng chênh lệch 0,8 F1 nằm sâu dưới ngưỡng phát hiện. Ô này chạy lại ở cỡ mẫu
lớn hơn để xem kết luận đó có đứng vững - nếu vẫn không tách được thì kết quả
âm được xác nhận ở mức chặt hơn.

In [ ]:
assert GPU, 'Ô này cần GPU'
SWEEP_N = 200 if SCALE != 'smoke' else 50
out = f'results/lambda_sweep_vimqa_{SWEEP_N}_{MTAG}.json'
if os.path.exists(out):
    print(f'[bỏ qua] đã có {out}')
else:
    print('=' * 62)
    print(f'### Quét λ và k - VimQA  n={SWEEP_N}')
    print('=' * 62, flush=True)
    rc, _ = run_stream(
        [sys.executable, '-u', 'src/scripts/run_lambda_sweep.py',
         '--dataset', 'vimqa', '--limit', str(SWEEP_N),
         '--reader', 'hf', '--reader-model', READER,
         *(['--load-4bit'] if LOAD_4BIT else []),
         '--out', out],
        f'results/log_lambda_{SWEEP_N}_{MTAG}.txt')
    if rc:
        print(f'!!! quét λ LỖI (mã {rc})')


## 13. Tải kết quả về

In [ ]:
import shutil, os

# Đóng gói kết quả. Colab tải về qua files.download; Kaggle không có API đó -
# file trong /kaggle/working tự xuất hiện ở tab Output khi commit notebook, nên
# chỉ cần đặt zip đúng chỗ.
zip_base = os.path.join(BASE, 'results')
shutil.make_archive(zip_base, 'zip', 'results')
zip_path = zip_base + '.zip'
print(f'{zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')

try:
    from google.colab import files          # noqa: F401
    files.download(zip_path)
except ImportError:
    print('Không phải Colab - lấy file ở:')
    print(f'  {zip_path}')
    print('Trên Kaggle: Save Version → Output → tải results.zip')
